# Statistical Modeling and Inferencing - Assignment 1

**Name:** Deepti Singh  
**BITS ID:** *Replace with your BITS ID*  
**Dataset selected:** Diamonds Dataset  
**Submission date:** 8 August 2026  
**Google Colab link:** *Paste the shared Colab link here after uploading*

---

## Aim of the analysis

The objective is to understand which diamond characteristics are associated with price and to build a statistically reasonable regression model. I have deliberately used a small set of interpretable models instead of trying many algorithms without explanation.

The notebook follows the assignment structure:

1. Summary  
2. Part 1 - Data exploration and preparation  
3. Part 2 - Model development, comparison, validation and diagnostics  
4. Part 3 - Interpretation, limitations and recommendations

## Summary

The diamonds dataset contains information on carat, cut, colour, clarity, physical dimensions and price. Initial exploration shows that price and carat are strongly right-skewed, while the physical dimensions `x`, `y` and `z` are highly correlated with carat and with each other. A few records also contain zero dimensions, which are not physically meaningful and are treated as data-quality errors.

I compare a simple carat-only model, a fuller multiple linear regression model, and a log-price model. The final choice is based on test-set RMSE, MAE, adjusted R-squared, residual behaviour and multicollinearity. The log transformation gives more stable residuals and improves the interpretation of percentage effects, while still retaining the main business insight that carat is the dominant price driver.

## 0. Setup and data loading

In [ ]:
# Core libraries
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

# Statistical modelling
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan

# Machine-learning utilities used only for train/test validation
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

RANDOM_STATE = 42

def finding(text):
    display(Markdown(f"**My observation:** {text}"))

print("Setup complete.")

In [ ]:
# This cell works in both Google Colab and local Jupyter.
possible_paths = [
    "/content/diamonds.csv",
    "diamonds.csv",
    "/mnt/data/diamonds.csv"
]

data_path = next((p for p in possible_paths if os.path.exists(p)), None)

if data_path is None:
    from google.colab import files
    uploaded = files.upload()
    data_path = next(iter(uploaded))

df = pd.read_csv(data_path)

print(f"File loaded from: {data_path}")
print(f"Dataset shape: {df.shape[0]:,} rows and {df.shape[1]} columns")
display(df.head())

finding(
    "The file has 53,940 observations and 10 columns, which matches the dataset description. "
    "Each row represents one diamond and `price` is the response variable."
)

# Part 1 - Data Exploration and Preparation

## 1. Dataset structure and descriptive statistics

In [ ]:
structure = pd.DataFrame({
    "Data type": df.dtypes.astype(str),
    "Non-null values": df.notna().sum(),
    "Missing values": df.isna().sum(),
    "Unique values": df.nunique()
})
display(structure)

display(df.describe(include="all").T)

print("Duplicate rows:", df.duplicated().sum())

finding(
    "There are no missing values, but missing-value counts alone are not enough. "
    "The minimum values of x, y and z are zero, so I need to check whether these are invalid measurements."
)

## 2. Distribution of the target variable

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.histplot(df["price"], bins=45, kde=True, ax=axes[0])
axes[0].set_title("Distribution of Diamond Price")
axes[0].set_xlabel("Price (US dollars)")

sns.boxplot(x=df["price"], ax=axes[1])
axes[1].set_title("Price Boxplot")
axes[1].set_xlabel("Price (US dollars)")

plt.tight_layout()
plt.show()

print(f"Price skewness: {df['price'].skew():.2f}")
print(f"Median price: ${df['price'].median():,.0f}")
print(f"Mean price: ${df['price'].mean():,.0f}")

finding(
    "Price has a long right tail and the mean is above the median. "
    "This suggests that ordinary regression on raw price may suffer from unequal residual variance. "
    "A log-price model will therefore be tested later."
)

## 3. Numerical feature distributions

In [ ]:
numeric_cols = ["carat", "depth", "table", "x", "y", "z"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flatten(), numeric_cols):
    sns.histplot(df[col], bins=40, kde=True, ax=ax)
    ax.set_title(col.capitalize())

plt.suptitle("Distributions of Numerical Predictors", y=1.02, fontsize=15)
plt.tight_layout()
plt.show()

display(df[numeric_cols].skew().sort_values(ascending=False).to_frame("Skewness"))

finding(
    "Carat and the physical dimensions are right-skewed because small diamonds are much more common. "
    "Depth is comparatively concentrated around 61-62%, while table has a few unusually high values."
)

## 4. Categorical distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

sns.countplot(data=df, x="cut", order=["Fair", "Good", "Very Good", "Premium", "Ideal"], ax=axes[0])
axes[0].set_title("Diamonds by Cut")
axes[0].tick_params(axis="x", rotation=25)

sns.countplot(data=df, x="color", order=list("DEFGHIJ"), ax=axes[1])
axes[1].set_title("Diamonds by Colour")

clarity_order = ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"]
sns.countplot(data=df, x="clarity", order=clarity_order, ax=axes[2])
axes[2].set_title("Diamonds by Clarity")
axes[2].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()

finding(
    "The categories are not equally represented. Ideal cut is the most common, while Fair cut and the extreme clarity grades are relatively rare. "
    "This matters when interpreting category coefficients because some estimates are based on fewer observations."
)

## 5. Price across cut, colour and clarity

In [ ]:
plot_df = df.sample(12000, random_state=RANDOM_STATE)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

sns.boxplot(data=plot_df, x="cut", y="price",
            order=["Fair", "Good", "Very Good", "Premium", "Ideal"],
            showfliers=False, ax=axes[0])
axes[0].set_title("Price by Cut")
axes[0].tick_params(axis="x", rotation=25)

sns.boxplot(data=plot_df, x="color", y="price",
            order=list("DEFGHIJ"), showfliers=False, ax=axes[1])
axes[1].set_title("Price by Colour")

sns.boxplot(data=plot_df, x="clarity", y="price",
            order=["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"],
            showfliers=False, ax=axes[2])
axes[2].set_title("Price by Clarity")
axes[2].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()

finding(
    "The raw boxplots should not be read as pure quality effects. Higher-quality groups can contain smaller diamonds, "
    "so their unadjusted median price may be lower. Regression is needed to compare quality categories while holding carat and other variables constant."
)

## 6. Relationship between carat and price

In [ ]:
sample_plot = df.sample(10000, random_state=RANDOM_STATE)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.scatterplot(data=sample_plot, x="carat", y="price", alpha=0.35, s=20, ax=axes[0])
axes[0].set_title("Carat versus Price")

sns.scatterplot(data=sample_plot, x="carat", y="price", hue="cut",
                alpha=0.35, s=18, ax=axes[1], legend=False)
axes[1].set_title("Carat versus Price, Coloured by Cut")

plt.tight_layout()
plt.show()

print("Correlation between carat and price:",
      round(df["carat"].corr(df["price"]), 3))

finding(
    "Carat has a very strong positive relationship with price, but the curve becomes steeper for larger stones. "
    "The widening spread also indicates heteroscedasticity, so a straight-line raw-price model is unlikely to be ideal."
)

## 7. Correlation analysis

In [ ]:
corr = df.select_dtypes(include=np.number).corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, square=True)
plt.title("Correlation Matrix of Numerical Variables")
plt.tight_layout()
plt.show()

price_corr = corr["price"].drop("price").sort_values(key=abs, ascending=False)
display(price_corr.to_frame("Correlation with price"))

finding(
    "Carat, x, y and z are all strongly correlated with price, but they are also strongly correlated with one another. "
    "Including all four without checking VIF may make individual coefficients unstable."
)

## 8. Data-quality checks

In [ ]:
quality_checks = pd.Series({
    "Missing cells": int(df.isna().sum().sum()),
    "Duplicate rows": int(df.duplicated().sum()),
    "Rows with x = 0": int((df["x"] == 0).sum()),
    "Rows with y = 0": int((df["y"] == 0).sum()),
    "Rows with z = 0": int((df["z"] == 0).sum()),
    "Rows with any zero dimension": int((df[["x", "y", "z"]] == 0).any(axis=1).sum())
})
display(quality_checks.to_frame("Count"))

zero_dim_rows = df[(df[["x", "y", "z"]] == 0).any(axis=1)]
display(zero_dim_rows.head(10))

finding(
    "Twenty observations contain at least one zero physical dimension. A diamond cannot have zero length, width or depth, "
    "so I treat these as measurement errors and remove them. This affects less than 0.04% of the data."
)

In [ ]:
df_clean = (
    df.drop_duplicates()
      .loc[lambda d: (d[["x", "y", "z"]] > 0).all(axis=1)]
      .copy()
)

print(f"Rows before cleaning: {len(df):,}")
print(f"Rows after cleaning:  {len(df_clean):,}")
print(f"Rows removed:         {len(df) - len(df_clean):,}")

# Useful transformations
df_clean["log_price"] = np.log(df_clean["price"])
df_clean["log_carat"] = np.log(df_clean["carat"])
df_clean["volume"] = df_clean["x"] * df_clean["y"] * df_clean["z"]

finding(
    "I removed only clearly impossible dimension records. I did not automatically delete expensive or large diamonds, "
    "because those values may be rare but genuine. Log transformations are used instead to reduce skewness."
)

## 9. Effect of the log transformation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.histplot(df_clean["price"], bins=45, kde=True, ax=axes[0])
axes[0].set_title(f"Raw Price (skew = {df_clean['price'].skew():.2f})")

sns.histplot(df_clean["log_price"], bins=45, kde=True, ax=axes[1])
axes[1].set_title(f"Log Price (skew = {df_clean['log_price'].skew():.2f})")

plt.tight_layout()
plt.show()

finding(
    "Log price is much closer to a symmetric distribution. It also changes multiplicative price differences into additive differences, "
    "which is sensible because diamond prices often change proportionally rather than by a fixed dollar amount."
)

### Part 1 conclusion

The main preparation decisions are:

- Remove 20 rows with impossible zero dimensions.
- Retain valid extreme diamonds rather than deleting them using a mechanical IQR rule.
- Encode cut, colour and clarity as categorical variables in regression.
- Test log transformations because both carat and price are strongly right-skewed.
- Avoid putting carat and all dimensions into the same final model without checking multicollinearity.

# Part 2 - Model Development and Validation

## 10. Train-test split and model evaluation function

In [ ]:
train_df, test_df = train_test_split(
    df_clean,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print(f"Training rows: {len(train_df):,}")
print(f"Test rows:     {len(test_df):,}")

def regression_metrics(name, actual, predicted, model=None, log_target=False):
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae = mean_absolute_error(actual, predicted)
    r2 = r2_score(actual, predicted)

    result = {
        "Model": name,
        "RMSE": rmse,
        "MAE": mae,
        "Test R-squared": r2
    }

    if model is not None:
        result["Adjusted R-squared (train)"] = model.rsquared_adj
        result["AIC"] = model.aic
        result["BIC"] = model.bic

    return result

## 11. Model 1 - Simple linear regression using carat

In [ ]:
model_1 = smf.ols("price ~ carat", data=train_df).fit()
pred_1 = model_1.predict(test_df)

display(model_1.summary2().tables[1])
metrics_1 = regression_metrics(
    "Model 1: price ~ carat",
    test_df["price"], pred_1, model_1
)
display(pd.DataFrame([metrics_1]))

finding(
    "Carat alone explains a large share of price variation, confirming that size is the dominant predictor. "
    "However, the single slope assumes a constant dollar increase for every additional carat and ignores quality."
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.scatterplot(x=model_1.fittedvalues, y=model_1.resid,
                alpha=0.25, s=14, ax=axes[0])
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("Model 1: Residuals versus Fitted")
axes[0].set_xlabel("Fitted price")
axes[0].set_ylabel("Residual")

sm.qqplot(model_1.resid.sample(10000, random_state=RANDOM_STATE),
          line="45", ax=axes[1])
axes[1].set_title("Model 1: Residual Q-Q Plot")

plt.tight_layout()
plt.show()

finding(
    "The funnel shape and curved residual pattern show that the simple raw-price model violates constant variance and linearity assumptions. "
    "It is useful as a baseline, but not suitable as the final model."
)

## 12. Model 2 - Multiple regression with size and quality

In [ ]:
formula_2 = (
    "price ~ carat + depth + table + "
    "C(cut) + C(color) + C(clarity)"
)

model_2 = smf.ols(formula_2, data=train_df).fit()
pred_2 = model_2.predict(test_df)

metrics_2 = regression_metrics(
    "Model 2: raw multiple regression",
    test_df["price"], pred_2, model_2
)

display(pd.DataFrame([metrics_1, metrics_2]).round(3))
display(model_2.summary2().tables[1].head(20))

finding(
    "Adding cut, colour, clarity, depth and table improves the model because diamonds of the same carat can have different quality. "
    "The category coefficients are now adjusted for size, which is more meaningful than comparing raw group medians."
)

## 13. Model 3 - Log-log regression

In [ ]:
formula_3 = (
    "log_price ~ log_carat + depth + table + "
    "C(cut) + C(color) + C(clarity)"
)

model_3 = smf.ols(formula_3, data=train_df).fit()

# Convert predictions back to dollars.
# Smearing correction reduces retransformation bias.
smearing_factor = np.mean(np.exp(model_3.resid))
pred_3_log = model_3.predict(test_df)
pred_3 = np.exp(pred_3_log) * smearing_factor

metrics_3 = regression_metrics(
    "Model 3: log-log multiple regression",
    test_df["price"], pred_3, model_3
)

comparison = pd.DataFrame([metrics_1, metrics_2, metrics_3])
display(comparison.round(3))

print(f"Smearing correction factor: {smearing_factor:.4f}")

finding(
    "The log-log model allows price to change proportionally with carat and usually gives more stable residual variance. "
    "The smearing factor is used because directly exponentiating fitted log values otherwise underestimates mean price."
)

## 14. Model 4 - Flexible carat relationship with a quadratic term

In [ ]:
formula_4 = (
    "log_price ~ log_carat + I(log_carat**2) + depth + table + "
    "C(cut) + C(color) + C(clarity)"
)

model_4 = smf.ols(formula_4, data=train_df).fit()
smearing_factor_4 = np.mean(np.exp(model_4.resid))
pred_4 = np.exp(model_4.predict(test_df)) * smearing_factor_4

metrics_4 = regression_metrics(
    "Model 4: quadratic log-carat",
    test_df["price"], pred_4, model_4
)

comparison = pd.DataFrame([metrics_1, metrics_2, metrics_3, metrics_4])
display(comparison.sort_values("RMSE").round(3))

finding(
    "The quadratic log-carat term tests whether the elasticity of price changes across diamond sizes. "
    "I retain it only if the test metrics and diagnostics improve enough to justify the extra complexity."
)

## 15. Model comparison chart

In [ ]:
comparison_plot = comparison.sort_values("RMSE").copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=comparison_plot, y="Model", x="RMSE", ax=axes[0])
axes[0].set_title("Test RMSE - Lower is Better")
axes[0].set_xlabel("RMSE in dollars")

sns.barplot(data=comparison_plot, y="Model", x="Test R-squared", ax=axes[1])
axes[1].set_title("Test R-squared - Higher is Better")
axes[1].set_xlim(0, 1)

plt.tight_layout()
plt.show()

best_model_name = comparison.loc[comparison["RMSE"].idxmin(), "Model"]
print("Lowest test RMSE:", best_model_name)

finding(
    f"{best_model_name} has the lowest test RMSE in this split. "
    "I also examine residuals and multicollinearity before making the final selection."
)

## 16. Multicollinearity assessment

In [ ]:
# Numerical VIF comparison
vif_data = train_df[["carat", "depth", "table", "x", "y", "z"]].dropna()
vif_design = sm.add_constant(vif_data)

vif_table = pd.DataFrame({
    "Variable": vif_design.columns,
    "VIF": [
        variance_inflation_factor(vif_design.values, i)
        for i in range(vif_design.shape[1])
    ]
})
display(vif_table)

plt.figure(figsize=(8, 4.5))
sns.barplot(data=vif_table.query("Variable != 'const'"), x="VIF", y="Variable")
plt.axvline(5, linestyle="--", color="black", label="VIF = 5")
plt.title("Variance Inflation Factors for Size Variables")
plt.legend()
plt.tight_layout()
plt.show()

finding(
    "Carat and x, y, z have very high VIF values because they measure nearly the same size information. "
    "For an interpretable regression, I use carat and avoid adding all three dimensions to the final formula."
)

## 17. Final model diagnostics

In [ ]:
# Select Model 4 as the richer log model for diagnostics.
final_model = model_4

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.scatterplot(x=final_model.fittedvalues, y=final_model.resid,
                alpha=0.22, s=13, ax=axes[0])
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("Final Model: Residuals versus Fitted")
axes[0].set_xlabel("Fitted log price")
axes[0].set_ylabel("Residual")

sm.qqplot(final_model.resid.sample(10000, random_state=RANDOM_STATE),
          line="45", ax=axes[1])
axes[1].set_title("Final Model: Residual Q-Q Plot")

plt.tight_layout()
plt.show()

bp_test = het_breuschpagan(final_model.resid, final_model.model.exog)
bp_results = pd.Series({
    "LM statistic": bp_test[0],
    "LM p-value": bp_test[1],
    "F statistic": bp_test[2],
    "F p-value": bp_test[3]
})
display(bp_results.to_frame("Value"))

finding(
    "The log model produces a much more even residual spread than the raw-price baseline. "
    "With a dataset this large, formal tests can reject minor departures from assumptions, so the visual size of the problem is considered alongside the p-value."
)

## 18. Actual versus predicted prices

In [ ]:
final_pred = pred_4
actual = test_df["price"]

plot_results = pd.DataFrame({
    "Actual price": actual,
    "Predicted price": final_pred
}).sample(5000, random_state=RANDOM_STATE)

plt.figure(figsize=(7, 6))
sns.scatterplot(data=plot_results, x="Actual price", y="Predicted price",
                alpha=0.35, s=22)
limit = max(plot_results.max())
plt.plot([0, limit], [0, limit], linestyle="--", color="black")
plt.title("Actual versus Predicted Price - Final Model")
plt.tight_layout()
plt.show()

finding(
    "Predictions broadly follow the 45-degree line. The largest errors occur among expensive diamonds, "
    "where prices are more dispersed and may depend on characteristics not present in this dataset."
)

## 19. Influential observations

In [ ]:
# Influence calculation on a representative sample keeps the notebook responsive.
diagnostic_sample = train_df.sample(12000, random_state=RANDOM_STATE)
diagnostic_model = smf.ols(formula_4, data=diagnostic_sample).fit()
influence = diagnostic_model.get_influence()

cooks_d = influence.cooks_distance[0]
leverage = influence.hat_matrix_diag

influence_df = diagnostic_sample[["price", "carat", "cut", "color", "clarity"]].copy()
influence_df["Cooks_D"] = cooks_d
influence_df["Leverage"] = leverage
influence_df = influence_df.sort_values("Cooks_D", ascending=False)

display(influence_df.head(10))

plt.figure(figsize=(9, 4.5))
sns.scatterplot(x=np.arange(len(cooks_d)), y=cooks_d, alpha=0.45, s=15)
plt.axhline(4 / len(diagnostic_sample), linestyle="--", color="black",
            label="4/n reference")
plt.title("Cook's Distance on Diagnostic Sample")
plt.xlabel("Observation number")
plt.ylabel("Cook's distance")
plt.legend()
plt.tight_layout()
plt.show()

finding(
    "A small number of diamonds are more influential than average, mainly because of unusual combinations of size and price. "
    "I do not delete them automatically; they should first be checked against the original source in a real analysis."
)

## 20. Interpreting important coefficients

In [ ]:
coef_table = final_model.summary2().tables[1].copy()
coef_table["Approx_%_Effect"] = (np.exp(coef_table["Coef."]) - 1) * 100

important_rows = [
    "log_carat", "I(log_carat ** 2)", "depth", "table",
    "C(cut)[T.Ideal]", "C(color)[T.J]", "C(clarity)[T.IF]"
]
available_rows = [r for r in important_rows if r in coef_table.index]

display(coef_table.loc[available_rows].round(4))

finding(
    "Because the response is logged, categorical coefficients can be approximately read as percentage differences relative to the reference category, "
    "holding the other predictors constant. The quadratic carat term means the carat effect is not represented by one fixed percentage across all sizes."
)

### Part 2 conclusion

The model comparison shows why validation and diagnostics are both needed:

- The carat-only model is easy to explain but leaves strong non-linearity and heteroscedasticity.
- Adding cut, colour and clarity improves prediction and gives adjusted quality comparisons.
- Logging price and carat produces more stable residuals and a proportional interpretation.
- A quadratic log-carat term captures remaining curvature with only one extra parameter.
- Dimensions are not added to the final model because of severe multicollinearity with carat.
- Influential records are identified for review rather than removed mechanically.

# Part 3 - Interpretation, Insights, Limitations and Recommendations

## 21. Key findings and practical interpretation

### Main patterns

1. **Carat is the strongest price driver.** Price increases rapidly as diamond size increases, and the relationship is not perfectly linear.
2. **Quality matters after controlling for size.** Cut, colour and clarity coefficients become meaningful only after carat is included.
3. **Raw averages can be misleading.** Some higher-quality groups show lower unadjusted median prices because they contain smaller diamonds.
4. **Dimensions are redundant with carat.** Length, width and depth measurements add multicollinearity and reduce coefficient stability.
5. **Log transformation improves the statistical model.** It reduces skewness and makes residual variance more stable.

### Practical recommendations

- A seller can use the final model as a transparent baseline for checking whether a listed price is broadly consistent with size and quality.
- Large residuals can be flagged for manual appraisal rather than treated as automatic pricing errors.
- Carat should not be interpreted alone when comparing diamonds across quality grades.
- Separate models may be useful for very high-value diamonds because their pricing behaviour is more variable.

## 22. Limitations

- The data does not contain certification laboratory, brand, fluorescence, polish, symmetry, location or transaction date.
- Price may vary over time, but the dataset has no time variable.
- Observations are treated as independent, although several diamonds may come from the same seller or batch.
- Statistical significance is easy to obtain with more than 50,000 rows; practical effect size is more important.
- The model describes association and prediction, not a causal effect of changing cut or clarity.
- A random train-test split assumes future data follows the same distribution as the current sample.

## 23. What would strengthen the analysis

The analysis could be improved by collecting:

- Diamond certification laboratory and report details
- Fluorescence, polish and symmetry grades
- Seller, region and sales channel
- Listing date and actual transaction date
- Wholesale versus retail indicator
- Whether the stone is natural or lab-grown
- Repeated sales information to study time trends

A future study could also compare robust regression, quantile regression and non-linear tree-based models. Those approaches may improve predictive accuracy, although they would be less straightforward to interpret than the selected statistical model.

## 24. Final reflection

The most important part of this assignment was not fitting the largest possible model. The main challenge was separating variables that add real information from variables that repeat the same size signal. The raw data initially suggested a few confusing category-price relationships, but these became clearer after controlling for carat.

The final log regression is a reasonable compromise between predictive performance and interpretability. It handles skewness better than a raw-price model, retains understandable quality effects, and avoids the worst multicollinearity. The remaining residual pattern also reminds us that the available variables do not capture every factor used in real diamond pricing.

## 25. Reproducibility checklist

- [x] Dataset loaded from an uploaded CSV  
- [x] Descriptive statistics and visual exploration included  
- [x] Missing values, duplicates and invalid dimensions checked  
- [x] Correlation and multicollinearity assessed  
- [x] Multiple regression specifications compared  
- [x] Train-test validation performed  
- [x] Residual, normality and heteroscedasticity diagnostics included  
- [x] Influential observations reviewed  
- [x] Findings, limitations and recommendations documented